# Day 1.1 — Embeddings 101 & Provider Tour

---

Today, in about 75 minutes, you'll answer three questions:

1. What is an **embedding**, in plain English?
2. How do you actually make one?
3. Which model / provider should you pick — and how do the popular ones compare?

**No math today.** We'll picture embeddings as arrows and let Python do the rest.


## 1. An embedding is a list of numbers that captures meaning

Imagine every sentence gets turned into an **arrow** pointing somewhere in space.

- Sentences with **similar meaning** get arrows pointing in **similar directions**.
- Sentences about **different topics** get arrows pointing in **different directions**.

That arrow is really just a list of numbers — usually 384, 768, 1536, or 3072 of them. That list is called an **embedding vector**.

| Sentence | Embedding (first 4 numbers of ~384) |
|---|---|
| "The cat sat on the mat" | `[0.021, -0.117,  0.043,  0.088, ...]` |
| "A kitten rested on the rug" | `[0.019, -0.110,  0.041,  0.091, ...]` |
| "Bitcoin hit an all-time high" | `[-0.204,  0.056, -0.172,  0.011, ...]` |

Notice how the first two vectors look similar (cats + resting) and the third looks totally different (finance). That "look similar" is the whole trick — it's how semantic search, RAG, and recommendations work under the hood.


## 2. Why embeddings matter for AI engineering

Before embeddings, computers matched text by **exact keywords**. If you searched for "car" you'd miss documents that said "vehicle" or "automobile."

With embeddings:
- "car", "vehicle", and "automobile" all point in similar directions → all get found.
- Search understands **meaning**, not just letters.

This one idea powers:
- **Semantic search** (this section's capstone)
- **RAG** — giving an LLM relevant docs before it answers (Section 6)
- **Recommendations** — "users who liked X also liked Y"
- **Deduplication** — finding near-duplicates in a dataset


## 3. Let's make our first embedding

We'll use `sentence-transformers` — free, open-source, runs on your laptop. First install:


In [ ]:
!pip install sentence-transformers --quiet

In [3]:
from sentence_transformers import SentenceTransformer

# First run downloads ~90 MB. Cached after that.
model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "The cat sat on the mat",
    "A kitten rested on the rug",
    "Bitcoin hit an all-time high",
]

vectors = model.encode(sentences)
print("Shape:", vectors.shape)           # (3 sentences, 384 numbers each)
print("First 5 numbers of sentence 1:", vectors[0][:5])


Shape: (3, 384)
First 5 numbers of sentence 1: [ 0.1304018  -0.01187016 -0.02811698  0.05123864 -0.05597447]


**What just happened:** `model.encode()` turned each sentence into a 384-number vector. That's it — that's an embedding.

The model is called `all-MiniLM-L6-v2`. It's small (~90 MB), fast, and free — a great starting point.


## 4. Popular embedding models — the shortlist

You don't need to know 50 models. In practice, engineers pick from a short list:

| Model | Type | Dims | Cost | When to use |
|---|---|---|---|---|
| `all-MiniLM-L6-v2` (Sentence Transformers) | Open source | 384 | Free | Default. Fast, small, good enough for most tasks. |
| `BAAI/bge-small-en-v1.5` | Open source | 384 | Free | Slightly better quality than MiniLM. Same speed class. |
| `text-embedding-3-small` (OpenAI) | Hosted API | 1536 | $0.02 / 1M tokens | When you want top quality without hosting anything. |
| `text-embedding-3-large` (OpenAI) | Hosted API | 3072 | $0.13 / 1M tokens | Best-in-class quality. Overkill for most starter projects. |

There's a public leaderboard called **MTEB** ([huggingface.co/spaces/mteb/leaderboard](https://huggingface.co/spaces/mteb/leaderboard)) that ranks embedding models — you don't need to memorize it, just know it exists for when you want to shop around.

Rule of thumb for freshers: **start with MiniLM. Upgrade only if quality is bad.**


## 5. Provider comparison — same sentence, three models

Let's embed the same sentence with three different models and see what changes.


In [4]:
# Compare two open-source models locally
from sentence_transformers import SentenceTransformer

sentence = "The Eiffel Tower is in Paris."

for model_name in ["all-MiniLM-L6-v2", "BAAI/bge-small-en-v1.5"]:
    m = SentenceTransformer(model_name)
    v = m.encode(sentence)
    print(f"{model_name:35s} -> {len(v)} dims, first 3: {v[:3]}")


all-MiniLM-L6-v2                    -> 384 dims, first 3: [0.07110152 0.03712863 0.0319643 ]
BAAI/bge-small-en-v1.5              -> 384 dims, first 3: [-0.03603699 -0.00615911  0.02105932]


In [ ]:
# OPTIONAL — OpenAI comparison. Skip if you don't have an API key.
import os
# from openai import OpenAI
# client = OpenAI()  # reads OPENAI_API_KEY
# resp = client.embeddings.create(model="text-embedding-3-small", input=sentence)
# v = resp.data[0].embedding
# print(f"text-embedding-3-small           -> {len(v)} dims, first 3: {v[:3]}")

print("Uncomment the block above once you have OPENAI_API_KEY set.")


**Notice:** the vectors from different models are **not comparable**. A MiniLM vector and a BGE vector live in different spaces — you can't mix them.

**Rule:** pick one embedding model per project. Re-embedding everything is annoying and expensive.


## 6. Cost & size — the practical view

If you embed 10,000 short documents (say ~50 tokens each = 500,000 tokens total):

| Model | Where it runs | Time on your laptop | Cost |
|---|---|---|---|
| MiniLM | Your CPU | ~30–60 seconds | $0 |
| BGE-small | Your CPU | ~30–60 seconds | $0 |
| OpenAI 3-small | API call | ~few seconds | $0.01 |
| OpenAI 3-large | API call | ~few seconds | $0.065 |

For a fresher project? MiniLM. For a production app where quality matters more than $10/month? OpenAI 3-small. That's the decision, most of the time.


## Recap

- An **embedding** is a list of numbers that captures the meaning of text.
- Similar meanings → similar vectors. That's the whole magic.
- `sentence-transformers` gives you free, open-source embeddings on your laptop.
- Start with **MiniLM**. Compare with BGE or OpenAI only if you need better quality.
- Vectors from different models don't mix — pick one and stick with it.

**Next class:** we'll take these vectors and actually **search** with them.
